# gnomAD v4.1: allele frequency across the human genome

gnomAD v4.1 genomes contain hundreds of millions of short variants.
This notebook uses the official tabix indexes to read evenly spaced
windows from all 22 autosomes without downloading the multi-gigabyte
chromosome VCFs in full. XY receives the sampled variants as one dense
scatter and keeps the source rows available as the view refines.

`GNOMAD_WINDOWS` and `GNOMAD_VARIANTS_PER_WINDOW` control the sample.
Set `GNOMAD_CHROMOSOMES=22` for a quick first run.

**Source:** [gnomAD v4.1 release](https://gnomad.broadinstitute.org/news/2024-04-gnomad-v4-1/)
and the public Google Cloud mirror under
`gs://gcp-public-data--gnomad/release/4.1/`.

Install beside XY with `python -m pip install numpy pysam requests xy`.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pysam
import requests

import xy

DATA_DIR = Path(os.getenv("XY_REAL_WORLD_DATA", "data")) / "gnomad"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CHROMOSOME_LENGTHS = {
    1: 248_956_422,
    2: 242_193_529,
    3: 198_295_559,
    4: 190_214_555,
    5: 181_538_259,
    6: 170_805_979,
    7: 159_345_973,
    8: 145_138_636,
    9: 138_394_717,
    10: 133_797_422,
    11: 135_086_622,
    12: 133_275_309,
    13: 114_364_328,
    14: 107_043_718,
    15: 101_991_189,
    16: 90_338_345,
    17: 83_257_441,
    18: 80_373_285,
    19: 58_617_616,
    20: 64_444_167,
    21: 46_709_983,
    22: 50_818_468,
}
chromosomes = [
    int(value)
    for value in os.getenv(
        "GNOMAD_CHROMOSOMES",
        ",".join(str(value) for value in CHROMOSOME_LENGTHS),
    ).split(",")
]
unknown = set(chromosomes).difference(CHROMOSOME_LENGTHS)
if unknown:
    raise ValueError(f"unknown autosomes: {sorted(unknown)}")

windows_per_chromosome = int(os.getenv("GNOMAD_WINDOWS", "8"))
variants_per_window = int(os.getenv("GNOMAD_VARIANTS_PER_WINDOW", "20000"))
window_width = int(os.getenv("GNOMAD_WINDOW_BP", "2000000"))
if min(windows_per_chromosome, variants_per_window, window_width) <= 0:
    raise ValueError("gnomAD window controls must be positive")

In [ ]:
genomic_position_parts = []
allele_frequency_parts = []
chromosome_parts = []

offsets = {}
running_offset = 0
for chromosome, length in CHROMOSOME_LENGTHS.items():
    offsets[chromosome] = running_offset
    running_offset += length

base_url = "https://storage.googleapis.com/gcp-public-data--gnomad/release/4.1/vcf/genomes"
for chromosome in chromosomes:
    length = CHROMOSOME_LENGTHS[chromosome]
    centers = (
        (np.arange(windows_per_chromosome, dtype=np.float64) + 0.5)
        * length
        / windows_per_chromosome
    )
    starts = np.clip(
        centers - window_width / 2,
        0,
        max(0, length - window_width),
    ).astype(np.int64)
    url = f"{base_url}/gnomad.genomes.v4.1.sites.chr{chromosome}.vcf.bgz"
    index_path = DATA_DIR / f"gnomad.genomes.v4.1.sites.chr{chromosome}.vcf.bgz.tbi"
    if not index_path.exists():
        response = requests.get(f"{url}.tbi", timeout=120)
        response.raise_for_status()
        partial = index_path.with_suffix(".tbi.part")
        partial.write_bytes(response.content)
        partial.replace(index_path)
    positions = []
    frequencies = []
    with pysam.VariantFile(url, index_filename=str(index_path)) as variants:
        for start in starts:
            kept = 0
            records = variants.fetch(
                f"chr{chromosome}",
                int(start),
                int(start + window_width),
            )
            for record in records:
                allele_frequencies = record.info.get("AF")
                if allele_frequencies is None:
                    continue
                for frequency in allele_frequencies:
                    if frequency is None or not 0 < frequency <= 1:
                        continue
                    positions.append(offsets[chromosome] + record.pos)
                    frequencies.append(float(frequency))
                    kept += 1
                    if kept >= variants_per_window:
                        break
                if kept >= variants_per_window:
                    break

    genomic_position_parts.append(np.asarray(positions, dtype=np.float64))
    allele_frequency_parts.append(np.asarray(frequencies, dtype=np.float64))
    chromosome_parts.append(np.full(len(positions), chromosome, dtype=np.float64))
    print(f"chr{chromosome}: {len(positions):,} variants")

genomic_position = np.concatenate(genomic_position_parts)
allele_frequency = np.concatenate(allele_frequency_parts)
chromosome_number = np.concatenate(chromosome_parts)
print(f"{genomic_position.size:,} variants total")

In [ ]:
tick_values = [
    offsets[chromosome] + CHROMOSOME_LENGTHS[chromosome] / 2 for chromosome in CHROMOSOME_LENGTHS
]
tick_labels = [str(chromosome) for chromosome in CHROMOSOME_LENGTHS]
chromosome_boundaries = [offsets[chromosome] for chromosome in range(2, 23)]

x_axis_style = {
    "axis_color": "#8094a5",
    "axis_width": 1.0,
    "grid_opacity": 0,
    "label_color": "#19364b",
    "label_size": 12,
    "tick_color": "#8094a5",
    "tick_label_color": "#40596a",
    "tick_label_size": 10,
    "tick_length": 4,
    "tick_width": 0.8,
}
y_axis_style = {
    **x_axis_style,
    "grid_color": "#b9cbd6",
    "grid_dash": "dotted",
    "grid_opacity": 0.78,
    "grid_width": 0.8,
}

chart = xy.scatter_chart(
    xy.scatter(
        genomic_position,
        allele_frequency,
        color=chromosome_number,
        color_domain=(-8, 22),
        colormap="pubu",
        size=1.0,
        opacity=0.7,
        density=True,
    ),
    *[
        xy.vline(
            boundary,
            color="#9db2c0",
            width=0.7,
            opacity=0.5,
            style={"dash": "2,5"},
        )
        for boundary in chromosome_boundaries
    ],
    xy.text(
        0.068,
        0.965,
        "gnomAD v4.1  /  ALLELE-FREQUENCY ATLAS",
        dx=0,
        dy=0,
        color="#102f44",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 21,
            "font_weight": 750,
            "letter_spacing": "0.02em",
            "vertical_align": "top",
        },
    ),
    xy.text(
        0.068,
        0.918,
        (
            f"{genomic_position.size:,} VARIANTS  ·  {len(chromosomes)} AUTOSOMES  ·  "
            f"{windows_per_chromosome} INDEXED WINDOWS EACH"
        ),
        dx=0,
        dy=0,
        color="#597487",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 10.5,
            "font_weight": 600,
            "letter_spacing": "0.085em",
            "vertical_align": "top",
        },
    ),
    xy.text(
        0.918,
        0.953,
        "AF  ·  LOG₁₀ SCALE",
        dx=0,
        dy=0,
        color="#087e8b",
        anchor="end",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 10.5,
            "font_weight": 700,
            "letter_spacing": "0.09em",
            "vertical_align": "top",
        },
    ),
    xy.text(
        0.49,
        0.035,
        "CHROMOSOME",
        dx=0,
        dy=0,
        color="#19364b",
        anchor="middle",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 12,
            "font_weight": 650,
            "letter_spacing": "0.055em",
        },
    ),
    xy.x_axis(
        tick_values=tick_values,
        tick_labels=tick_labels,
        style=x_axis_style,
    ),
    xy.y_axis(
        label="ALTERNATE ALLELE FREQUENCY",
        label_offset=-18,
        type_="log",
        domain=(1e-7, 1.0),
        tick_values=[1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0],
        tick_labels=["10⁻⁷", "10⁻⁶", "10⁻⁵", "10⁻⁴", "10⁻³", "10⁻²", "10⁻¹", "1"],
        style=y_axis_style,
    ),
    xy.colorbar(
        title="AUTOSOME",
        ticks=[1, 6, 11, 16, 22],
        style={
            "background": "#ffffffcc",
            "border": "1px solid #d7e4eb",
            "border_radius": 6,
            "color": "#40596a",
            "padding": "7px 8px",
        },
    ),
    xy.theme(
        background="#f2f8fb",
        plot_background="#fbfdfe",
        text_color="#19364b",
        grid_color="#b9cbd6",
        axis_color="#8094a5",
        crosshair_color="#087e8b",
        selection_color="#075985",
        selection_fill="#0891b226",
    ),
    styles={
        "axis_title": {"font_weight": 650, "letter_spacing": "0.055em"},
        "colorbar_tick": {
            "color": "#40596a",
            "font_size": 10,
            "font_variant_numeric": "tabular-nums",
        },
        "colorbar_title": {
            "color": "#19364b",
            "font_size": 10,
            "font_weight": 700,
            "letter_spacing": "0.07em",
        },
        "tick_label": {"font_variant_numeric": "tabular-nums"},
    },
    padding=(92, 110, 88, 104),
    width=1150,
    height=620,
)
print(chart.memory_report()["canonical_bytes"], "canonical bytes")
chart